In [2]:
import numpy as np
import cupy as cp
from cupyx.scipy import ndimage
from tanalysis import improcess
from scipy import ndimage as ndi
import tifffile as tiff
from skimage.measure import label, regionprops
from skimage import exposure, morphology, filters
import matplotlib.pyplot as plt

In [ ]:
def LoG(sigma_x, sigma_y, sigma_z):
    n = 7
    z,y,x = cp.ogrid[-n//2:n//2+1, -n//2:n//2+1, -n//2:n//2+1]
    z_filter = cp.exp(-(z*z/(2*sigma_z**2)))
    y_filter = cp.exp(-(y*y/(2*sigma_y**2)))
    x_filter = cp.exp(-(x*x/(2*sigma_x**2)))
    final_filter = (1/(sigma_x*sigma_y*sigma_z*(2*np.pi)**(3/2)))*((1-x*x/(sigma_x**2))/(sigma_x**2)+(1-y*y/(sigma_y**2))/(sigma_y**2)+(1-z*z/(sigma_z**2))/(sigma_z**2))*(z_filter*x_filter*y_filter)
    return final_filter

In [ ]:
def LoG_convolve(img, sigmas_x, sigma_y, sigma_z):
    filter_log = LoG(sigmas_x, sigma_y, sigma_z)
    image = ndimage.convolve(img, filter_log)
    image = cp.square(image)
    return cp.asarray(image)

In [3]:
def fit_gaussian(sigma_x, sigma_y, n=7, amplitude=10):
    y,x = np.ogrid[-n//2:n//2+1, -n//2:n//2+1]
    y_filter = np.exp(-(y*y/(2*sigma_y**2)))
    x_filter = np.exp(-(x*x/(2*sigma_x**2)))
    final_filter = amplitude*(x_filter*y_filter)/(sigma_x*sigma_y*(2*np.pi)**(3/2))
    return final_filter

In [7]:
image = np.ones((100,100), dtype=np.double)
gauss_filt = fit_gaussian(10,10, n=99)
gauss = image * gauss_filt
gauss = gauss/np.max(gauss)
log = ndi.laplace(gauss)
print(np.min(log))

-0.01995008322927072


In [ ]:
def label_cells(image:np.ndarray, sigmas:list, th:float):
    '''
    '''
    X,Y,Z = image.shape
    og_img = exposure.equalize_adapthist(image, kernel_size=(X//5, Y//5, Z))
    og_img = np.double(og_img)
    img = cp.array(og_img)

    gaussian = fit_gaussian(*sigmas)
    img = ndimage.convolve(img, gaussian)
    th1 = cp.max(img)*th #####################
    img_th1 = img>th1

    img = LoG_convolve(img, *sigmas)
    th2 = cp.max(img)*th #####################
    img_th2 = img>th2

    img_th1 = cp.asnumpy(img_th1)
    img_th2 = cp.asnumpy(img_th2)

    img = img_th1 ^ img_th2
    img = morphology.dilation(img)
    img = morphology.erosion(img)
    img = morphology.remove_small_holes(img)
    img_r1 = morphology.remove_small_objects(img)

    img = cp.asarray(np.float16(img_r1))
    img = LoG_convolve(img, *sigmas)
    img = cp.asnumpy(img)
    th3 = np.max(img)*th
    img_th3 = img>th3
    img_r2 = img_r1 ^ img_th3
    img = morphology.erosion(img_r2)
    img = morphology.remove_small_objects(img)
    img = label(img)
    return img

In [ ]:
dirname = r"C:\Users\pcanaleta\Documents\Cellpose_segmentation\EXP.HD6.Chips\EXP.HD6.1.2.Test\EXP.HD6.1.2.Test.0\24h\Stitched\24h_chem20_thunder-0.tiff"
imgs, names, info = improcess.imread(dirname, False, False)

In [ ]:
T,Z,Y,X = imgs[0].shape
result = np.empty(shape=(T,X,Y,Z), dtype=np.int8)
t = 0

for frame in imgs[0]:
    result[t] = label_cells(frame.T, sigmas=(1,1,0.5), th=0.1)
    t+=1

In [ ]:
Z,Y,X = imgs[0].shape
result = np.empty(shape=(Z,Y,X), dtype=np.int8)

for img in imgs:
    th = np.max(img)*0.1
    result = img>th
    result = morphology.dilation(result)
    result = morphology.erosion(result)
    result = morphology.remove_small_objects(result)
    result = morphology.remove_small_holes(result)
    lab_img = label(result)

In [ ]:
regions = regionprops(lab_img)
for props in regions:
    print(props.bbox)
    print(props.axis_major_length)
    print(props.centroid)
    print(props.slice)

In [ ]:
tiff.imwrite(r"C:\Users\pcanaleta\Documents\Img_Processing\Fibros\Processed\24h_Fibros_T0.tiff", 
            np.uint8(result), 
            imagej=True,
            metadata={
                'axes':"TZYX"
            })